In [18]:
import pandas as pd
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

In [19]:
housing = pd.read_csv(r"C:\Users\PGCP-AI.STUDENTSDC\Desktop\Machine_Learning\Datasets\Housing.csv")
housing

,price,lotsize,bedrooms,bathrms,stories,driveway,recroom,fullbase,gashw,airco,garagepl,prefarea
0,42000.0,5850,3,1,2,yes,no,yes,no,no,1,no
1,38500.0,4000,2,1,1,yes,no,no,no,no,0,no
2,49500.0,3060,3,1,1,yes,no,no,no,no,0,no
3,60500.0,6650,3,1,2,yes,yes,no,no,no,0,no
4,61000.0,6360,2,1,1,yes,no,no,no,no,0,no
...,...,...,...,...,...,...,...,...,...,...,...,...
541,91500.0,4800,3,2,4,yes,yes,no,no,yes,0,no
542,94000.0,6000,3,2,4,yes,no,no,no,yes,0,no
543,103000.0,6000,3,2,4,yes,yes,no,no,yes,1,no
544,105000.0,6000,3,2,2,yes,yes,no,no,yes,1,no


In [20]:
intervals = [(0,50000),(50000,100000),(100000,150000),(150000,200000)]
bins = pd.IntervalIndex.from_tuples(intervals)
housing['price_slab'] = pd.cut(housing['price'],bins)
intervals = [(0,5000),(5000,10000),(10000,15000),(15000,20000)]
bins = pd.IntervalIndex.from_tuples(intervals)
housing['area_slab'] = pd.cut(housing['lotsize'],bins)
housing.drop(['price','lotsize'],axis=1,inplace=True)

In [21]:
housing

,bedrooms,bathrms,stories,driveway,recroom,fullbase,gashw,airco,garagepl,prefarea,price_slab,area_slab
0,3,1,2,yes,no,yes,no,no,1,no,"(0, 50000]","(5000, 10000]"
1,2,1,1,yes,no,no,no,no,0,no,"(0, 50000]","(0, 5000]"
2,3,1,1,yes,no,no,no,no,0,no,"(0, 50000]","(0, 5000]"
3,3,1,2,yes,yes,no,no,no,0,no,"(50000, 100000]","(5000, 10000]"
4,2,1,1,yes,no,no,no,no,0,no,"(50000, 100000]","(5000, 10000]"
...,...,...,...,...,...,...,...,...,...,...,...,...
541,3,2,4,yes,yes,no,no,yes,0,no,"(50000, 100000]","(0, 5000]"
542,3,2,4,yes,no,no,no,yes,0,no,"(50000, 100000]","(5000, 10000]"
543,3,2,4,yes,yes,no,no,yes,1,no,"(100000, 150000]","(5000, 10000]"
544,3,2,2,yes,yes,no,no,yes,1,no,"(100000, 150000]","(5000, 10000]"


In [22]:
from sklearn.preprocessing import OneHotEncoder
import numpy as np

ohc = OneHotEncoder(sparse_output=False, dtype=np.bool_).set_output(transform='pandas')
df = ohc.fit_transform(housing)

In [23]:
df

,bedrooms_1,bedrooms_2,bedrooms_3,bedrooms_4,bedrooms_5,bedrooms_6,bathrms_1,bathrms_2,bathrms_3,bathrms_4,...,prefarea_no,prefarea_yes,"price_slab_(0, 50000]","price_slab_(50000, 100000]","price_slab_(100000, 150000]","price_slab_(150000, 200000]","area_slab_(0, 5000]","area_slab_(5000, 10000]","area_slab_(10000, 15000]","area_slab_(15000, 20000]"
0,False,False,True,False,False,False,True,False,False,False,...,True,False,True,False,False,False,False,True,False,False
1,False,True,False,False,False,False,True,False,False,False,...,True,False,True,False,False,False,True,False,False,False
2,False,False,True,False,False,False,True,False,False,False,...,True,False,True,False,False,False,True,False,False,False
3,False,False,True,False,False,False,True,False,False,False,...,True,False,False,True,False,False,False,True,False,False
4,False,True,False,False,False,False,True,False,False,False,...,True,False,False,True,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
541,False,False,True,False,False,False,False,True,False,False,...,True,False,False,True,False,False,True,False,False,False
542,False,False,True,False,False,False,False,True,False,False,...,True,False,False,True,False,False,False,True,False,False
543,False,False,True,False,False,False,False,True,False,False,...,True,False,False,False,True,False,False,True,False,False
544,False,False,True,False,False,False,False,True,False,False,...,True,False,False,False,True,False,False,True,False,False


In [24]:
from ipywidgets import interact
import ipywidgets as widgets

In [25]:
def gen_rules(min_support,min_confidence):
    itemsets = apriori(df,min_support=min_support,use_colnames=True)
    rules = association_rules(itemsets,metric='confidence',min_threshold=min_confidence)
    rules = rules[['antecedents','consequents','support','confidence','lift']]
    relv_rules = rules[rules['lift']>1]
    return relv_rules.sort_values(by=['confidence','lift','support'],ascending=False)

In [ ]:
interact(gen_rules,
         min_support = widgets.FloatSlider(value=0.01,min=0.01,max=0.1,step=0.01,description='Min Support:'),
         min_confidence = widgets.FloatSlider(value=0.01,min=0,max=1,step=0.01,description='Min Confidence:'))

In [29]:
gen_rules(0.5,0.5)

,antecedents,consequents,support,confidence,lift
42,(garagepl_0),(gashw_no),0.534799,0.973333,1.020038
60,"(bathrms_1, driveway_yes)",(gashw_no),0.602564,0.964809,1.011105
8,(bathrms_1),(gashw_no),0.708791,0.962687,1.008881
169,"(bathrms_1, driveway_yes, recroom_no)",(gashw_no),0.503663,0.961538,1.007678
67,"(bathrms_1, recroom_no)",(gashw_no),0.602564,0.959184,1.005210
...,...,...,...,...,...
129,(driveway_yes),"(gashw_no, price_slab_(50000, 100000])",0.512821,0.597015,1.061792
89,(gashw_no),"(bathrms_1, prefarea_no)",0.554945,0.581574,1.004871
43,(gashw_no),(garagepl_0),0.534799,0.560461,1.020038
161,(gashw_no),"(prefarea_no, fullbase_no)",0.520147,0.545106,1.002113


# Exercise

In [65]:
df = pd.read_csv(r"C:\Users\PGCP-AI.STUDENTSDC\Desktop\Machine_Learning\Cases\human-resources-analytics\HR_comma_sep.csv")
df

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.10,0.77,6,247,4,0,1,0,sales,low
3,0.92,0.85,5,259,5,0,1,0,sales,low
4,0.89,1.00,5,224,5,0,1,0,sales,low
...,...,...,...,...,...,...,...,...,...,...
14990,0.40,0.57,2,151,3,0,1,0,support,low
14991,0.37,0.48,2,160,3,0,1,0,support,low
14992,0.37,0.53,2,143,3,0,1,0,support,low
14993,0.11,0.96,6,280,4,0,1,0,support,low


In [66]:
df.describe()

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years
count,14995.000000,14995.000000,14995.000000,14995.000000,14995.000000,14995.000000,14995.000000,14995.000000
mean,0.612890,0.716108,3.803001,201.050150,3.498166,0.144648,0.237879,0.021274
std,0.248615,0.171166,1.232266,49.943356,1.460262,0.351758,0.425799,0.144300
min,0.090000,0.360000,2.000000,96.000000,2.000000,0.000000,0.000000,0.000000
25%,0.440000,0.560000,3.000000,156.000000,3.000000,0.000000,0.000000,0.000000
50%,0.640000,0.720000,4.000000,200.000000,3.000000,0.000000,0.000000,0.000000
75%,0.820000,0.870000,5.000000,245.000000,4.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,7.000000,310.000000,10.000000,1.000000,1.000000,1.000000


In [68]:
def bin_col(col,n):
    binned = pd.cut(col,bins=n)
    return binned

In [69]:
df['satisfaction'] = bin_col(df['satisfaction_level'],5)
df['evaluation'] = bin_col(df['last_evaluation'],5)
df['avg_hours'] = bin_col(df['average_montly_hours'],5)
df['project'] = bin_col(df['number_project'],3)
df['yrs_in_comp'] = bin_col(df['time_spend_company'],3)
df.drop(['satisfaction_level','last_evaluation','average_montly_hours','number_project',
         'time_spend_company'],axis=1,inplace=True)

In [53]:
# intervals = [(0,0.3),(0.3,0.6),(0.6,1)]
# bins = pd.IntervalIndex.from_tuples(intervals)
# df['satisfaction_level_sal'] = pd.cut(df['satisfaction_level'],bins)
# df['last_evaluation_sal'] = pd.cut(df['last_evaluation'],bins)
# intervals = [(0,100),(100,200),(200,300),(300,350)]
# bins = pd.IntervalIndex.from_tuples(intervals)
# df['average_montly_hours_sal'] = pd.cut(df['average_montly_hours'],bins)
# df.drop(['average_montly_hours','last_evaluation','satisfaction_level'],axis=1,inplace=True)
# df

,number_project,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary,satisfaction_level_sal,last_evaluation_sal,average_montly_hours_sal
0,2,3,0,1,0,sales,low,"(0.3, 0.6]","(0.3, 0.6]","(100, 200]"
1,5,6,0,1,0,sales,medium,"(0.6, 1.0]","(0.6, 1.0]","(200, 300]"
2,6,4,0,1,0,sales,low,"(0.0, 0.3]","(0.6, 1.0]","(200, 300]"
3,5,5,0,1,0,sales,low,"(0.6, 1.0]","(0.6, 1.0]","(200, 300]"
4,5,5,0,1,0,sales,low,"(0.6, 1.0]","(0.6, 1.0]","(200, 300]"
...,...,...,...,...,...,...,...,...,...,...
14990,2,3,0,1,0,support,low,"(0.3, 0.6]","(0.3, 0.6]","(100, 200]"
14991,2,3,0,1,0,support,low,"(0.3, 0.6]","(0.3, 0.6]","(100, 200]"
14992,2,3,0,1,0,support,low,"(0.3, 0.6]","(0.3, 0.6]","(100, 200]"
14993,6,4,0,1,0,support,low,"(0.0, 0.3]","(0.6, 1.0]","(200, 300]"


In [70]:
df

,Work_accident,left,promotion_last_5years,Department,salary,satisfaction,evaluation,avg_hours,project,yrs_in_comp
0,0,1,0,sales,low,"(0.272, 0.454]","(0.488, 0.616]","(138.8, 181.6]","(1.995, 3.667]","(1.992, 4.667]"
1,0,1,0,sales,medium,"(0.636, 0.818]","(0.744, 0.872]","(224.4, 267.2]","(3.667, 5.333]","(4.667, 7.333]"
2,0,1,0,sales,low,"(0.0891, 0.272]","(0.744, 0.872]","(224.4, 267.2]","(5.333, 7.0]","(1.992, 4.667]"
3,0,1,0,sales,low,"(0.818, 1.0]","(0.744, 0.872]","(224.4, 267.2]","(3.667, 5.333]","(4.667, 7.333]"
4,0,1,0,sales,low,"(0.818, 1.0]","(0.872, 1.0]","(181.6, 224.4]","(3.667, 5.333]","(4.667, 7.333]"
...,...,...,...,...,...,...,...,...,...,...
14990,0,1,0,support,low,"(0.272, 0.454]","(0.488, 0.616]","(138.8, 181.6]","(1.995, 3.667]","(1.992, 4.667]"
14991,0,1,0,support,low,"(0.272, 0.454]","(0.359, 0.488]","(138.8, 181.6]","(1.995, 3.667]","(1.992, 4.667]"
14992,0,1,0,support,low,"(0.272, 0.454]","(0.488, 0.616]","(138.8, 181.6]","(1.995, 3.667]","(1.992, 4.667]"
14993,0,1,0,support,low,"(0.0891, 0.272]","(0.872, 1.0]","(267.2, 310.0]","(5.333, 7.0]","(1.992, 4.667]"


In [71]:
encoder = OneHotEncoder(sparse_output=False, dtype=bool).set_output(transform='pandas')
df_en = encoder.fit_transform(df)
df_en

,Work_accident_0,Work_accident_1,left_0,left_1,promotion_last_5years_0,promotion_last_5years_1,Department_IT,Department_RandD,Department_accounting,Department_hr,...,"avg_hours_(138.8, 181.6]","avg_hours_(181.6, 224.4]","avg_hours_(224.4, 267.2]","avg_hours_(267.2, 310.0]","project_(1.995, 3.667]","project_(3.667, 5.333]","project_(5.333, 7.0]","yrs_in_comp_(1.992, 4.667]","yrs_in_comp_(4.667, 7.333]","yrs_in_comp_(7.333, 10.0]"
0,True,False,False,True,True,False,False,False,False,False,...,True,False,False,False,True,False,False,True,False,False
1,True,False,False,True,True,False,False,False,False,False,...,False,False,True,False,False,True,False,False,True,False
2,True,False,False,True,True,False,False,False,False,False,...,False,False,True,False,False,False,True,True,False,False
3,True,False,False,True,True,False,False,False,False,False,...,False,False,True,False,False,True,False,False,True,False
4,True,False,False,True,True,False,False,False,False,False,...,False,True,False,False,False,True,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14990,True,False,False,True,True,False,False,False,False,False,...,True,False,False,False,True,False,False,True,False,False
14991,True,False,False,True,True,False,False,False,False,False,...,True,False,False,False,True,False,False,True,False,False
14992,True,False,False,True,True,False,False,False,False,False,...,True,False,False,False,True,False,False,True,False,False
14993,True,False,False,True,True,False,False,False,False,False,...,False,False,False,True,False,False,True,True,False,False


In [75]:
def gen_rules(min_support,min_confidence):
    itemsets = apriori(df_en,min_support=min_support,use_colnames=True)
    rules = association_rules(itemsets,metric='confidence',min_threshold=min_confidence)
    rules = rules[['antecedents','consequents','support','confidence','lift']]
    relv_rules = rules[rules['lift']>1]
    return relv_rules.sort_values(by=['confidence','lift','support'],ascending=False)

In [78]:
interact(gen_rules,
         min_support = widgets.FloatSlider(value=0.01,min=0,max=0.1,step=0.01,description='Min Support:'),
         min_confidence = widgets.FloatSlider(value=0.01,min=0,max=1,step=0.01,description='Min Confidence:'))

interactive(children=(FloatSlider(value=0.01, description='Min Support:', max=0.1, step=0.01), FloatSlider(val…

<function __main__.gen_rules(min_support, min_confidence)>